# Moving letters: teaching a model J and Z

J and Z are the only ASL letters you **draw in the air**, so one photo of the hand isn't enough. This notebook teaches a small model to read a whole **move** (one to four seconds of hand dots) and say **J**, **Z** or **other** (any move that isn't J or Z).

It's the same plan as the still-letter notebook (`train_letters.ipynb`):

1. Load your recordings
2. Cut them into moves
3. Turn each move into numbers
4. Train a model
5. Test it on the exam: moves it never studied
6. Save it for the website

## 1. Upload your recordings

Click the **folder icon** on the left, then upload every `nm-jz-….csv` file **and** `nm-moves-exam.csv` from `training/data` in your project. Then run this cell.

In [ ]:
import glob, json
import numpy as np
import pandas as pd
import tensorflow as tf

practice_files = sorted(glob.glob("nm-jz-*.csv"), key=lambda name: int(name.split("-")[-1].split(".")[0]))
EXAM_FILE = "nm-moves-exam.csv"

print(len(practice_files), "practice files:", practice_files)
print("Exam file uploaded:", EXAM_FILE in glob.glob("*.csv"))

## 2. The move cutter

A recording is just a long list of **frames** (one row = the 21 dots of your hand in one camera picture). The app decides where a move starts and ends: **speed up → a move starts; freeze → the move is over.**

This cell is the **same rules, with the same numbers,** as the app's `src/lib/asl/motion.ts`, written in Python. They have to match, so the model learns from moves cut exactly the way the website will cut them.

In [ ]:
# The same rules (and numbers) as src/lib/asl/motion.ts in the app.
TRACKED = [0, 4, 8, 12, 16, 20]  # the wrist and the five fingertips
START = 0.03        # a move starts when the hand goes faster than this (palm sizes per frame)...
STOP = 0.02         # ...and ends when it has been slower than this
STILL_FRAMES = 5    # ...for 5 frames in a row (a freeze)
MIN_FRAMES = 15     # shorter moves are just a twitch
MAX_FRAMES = 200    # longer moves are several things in a row
MIN_SIZE = 0.6      # smaller moves (in palm sizes) are a shaky hand
MAX_JUMP = 1.0      # a bigger jump in one frame means the dots jumped to another hand
GAP_JUMP = 0.3      # a still hand that suddenly jumps this far = a gap between two key presses
PRE_ROLL = 3        # frames kept from just before the move started
SMOOTH = 3          # average the speed over 3 frames (the camera sometimes repeats a frame)

def palm_size(hand):
    """Wrist (dot 0) to the bottom of the middle finger (dot 9), flat on the picture."""
    return np.hypot(hand[9, 0] - hand[0, 0], hand[9, 1] - hand[0, 1])

def hand_speed(previous, current):
    """How far the wrist and fingertips moved since the previous frame, in palm sizes."""
    palm = palm_size(current)
    if not palm:
        return 0.0
    steps = [np.hypot(current[i, 0] - previous[i, 0], current[i, 1] - previous[i, 1]) for i in TRACKED]
    return sum(steps) / len(TRACKED) / palm

def move_size(move):
    """The widest path drawn by any tracked dot, in palm sizes."""
    palm = np.mean([palm_size(hand) for hand in move])
    if not palm:
        return 0.0
    return max(np.hypot(*(move[:, i, :2].max(axis=0) - move[:, i, :2].min(axis=0))) for i in TRACKED) / palm

def cut_moves(frames):
    """Cuts a list of frames into moves. Returns (moves, what happened to each one)."""
    moves, results = [], []
    move, before, speeds = [], [], []
    previous, still, too_long, biggest_step = None, 0, False, 0.0

    def finish():
        nonlocal move, before, still, too_long, biggest_step
        frames_, too_long_, biggest_ = np.asarray(move), too_long, biggest_step
        move, before, still, too_long, biggest_step = [], [], 0, False, 0.0
        if too_long_: results.append("too long")
        elif not len(frames_): pass
        elif biggest_ > MAX_JUMP: results.append("jumped")
        elif len(frames_) < MIN_FRAMES: results.append("too short")
        elif move_size(frames_) < MIN_SIZE: results.append("too small")
        else:
            moves.append(frames_)
            results.append("saved")

    for hand in frames:
        step = hand_speed(previous, hand) if previous is not None else 0.0
        previous = hand
        moving = len(move) > 0 or too_long
        if not moving and step > GAP_JUMP:  # a gap in the recording: start afresh
            speeds, before = [], [hand]
            continue
        speeds = (speeds + [step])[-SMOOTH:]
        speed = sum(speeds) / len(speeds)
        if not moving:
            if speed > START:  # a move starts
                move, still, biggest_step = before + [hand], 0, step
            else:
                before = (before + [hand])[-PRE_ROLL:]
            continue
        biggest_step = max(biggest_step, step)
        if not too_long:
            move.append(hand)
        if len(move) > MAX_FRAMES:
            too_long, move = True, []
        still = still + 1 if speed < STOP else 0
        if still >= STILL_FRAMES:  # frozen long enough: the move is over
            finish()
    finish()  # the end of the recording also ends a move
    return moves, results

## 3. Cut every recording into moves

The `label` column says which key you were holding (J, Z, or `other` for the space bar). Every stretch of rows with the same key is cut into moves.

In [ ]:
def moves_in(file):
    """Cuts one recording into moves. Returns a list of (label, move) pairs."""
    table = pd.read_csv(file)
    labels = table["label"].values
    frames = table.drop(columns="label").values.reshape(-1, 21, 3)  # each row → 21 dots × (x, y, z)
    found, start = [], 0
    for i in range(1, len(labels) + 1):
        if i == len(labels) or labels[i] != labels[start]:  # where one key's rows end
            moves, _ = cut_moves(list(frames[start:i]))
            found += [(labels[start], move) for move in moves]
            start = i
    return found

practice = [pair for file in practice_files for pair in moves_in(file)]
print(pd.Series([label for label, _ in practice]).value_counts())

## 4. Throw out broken J's and Z's

A few "J" and "Z" moves aren't really letters: your hand going back to the start, or the last line of a Z after a pause. Both letters **end lower than they start**, so if the fingertip doesn't finish at least a third of a palm lower, that move is dropped. (The pinky draws a J; the index finger draws a Z.)

In [ ]:
def real_letter(label, move):
    """False for a 'J' or 'Z' that isn't really one (both letters end lower than they start)."""
    if label not in ("J", "Z"):
        return True
    tip = 20 if label == "J" else 8  # J is drawn with the pinky, Z with the index finger
    palm = np.mean([palm_size(hand) for hand in move])
    drop = (move[-1, tip, 1] - move[0, tip, 1]) / palm  # in camera pictures, y grows downwards
    return drop >= 0.3

before = len(practice)
practice = [(label, move) for label, move in practice if real_letter(label, move)]
print("Dropped", before - len(practice), "broken moves. Left:")
print(pd.Series([label for label, _ in practice]).value_counts())

## 5. Look at some moves

Each picture is the path of the fingertip during one move, shown like a mirror (the way you see yourself). Green = start, red = end.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 6, figsize=(12, 6))
for row, name in enumerate(["J", "Z", "other"]):
    examples = [move for label, move in practice if label == name][-6:]
    for ax, move in zip(axes[row], examples):
        tip = move[:, 20 if name == "J" else 8]  # the pinky for J, the index finger otherwise
        ax.plot(-tip[:, 0], tip[:, 1])  # minus x: show it like a mirror, the way you see yourself
        ax.plot(-tip[0, 0], tip[0, 1], "go")
        ax.plot(-tip[-1, 0], tip[-1, 1], "rs")
        ax.invert_yaxis()
        ax.set_aspect("equal")
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(name)
plt.tight_layout()
plt.show()

## 6. Turn each move into numbers

A model needs every example to be the same size, but a J might take 50 frames and a slow Z 150. So for every move:

1. **Stretch or squash it to exactly 24 frames**, like resizing every video to the same length.
2. **Start at 0,0,0:** measure every dot from where the wrist was at the start, so it doesn't matter where you were in the picture.
3. **Divide by the palm size**, so a hand near the camera and a hand far away look the same.
4. **Mirror left hands** so they look like right hands (the same trick as the still letters).

Each move becomes a table of **24 rows (frames) × 63 numbers (21 dots × x, y, z)**. The website does exactly the same in `src/lib/asl/moveModel.ts`, and a test there checks that both give the same numbers.

In [ ]:
FRAMES = 24
LABELS = ["J", "Z", "other"]

def resample(move, frames=FRAMES):
    """Stretches or squashes a move to exactly `frames` frames, blending neighbouring frames."""
    t = np.linspace(0, len(move) - 1, frames)  # where each new frame falls in the old move
    before = np.floor(t).astype(int)
    after = np.minimum(before + 1, len(move) - 1)
    mix = (t - before)[:, None, None]  # how far it is from the frame before to the frame after
    return move[before] * (1 - mix) + move[after] * mix

def move_features(move):
    """The same recipe as moveFeatures() in src/lib/asl/moveModel.ts."""
    m = resample(np.asarray(move, dtype=np.float64))
    m = m - m[0, 0]  # the wrist at the start becomes 0,0,0
    palm = np.mean(np.linalg.norm(m[:, 9] - m[:, 0], axis=1))
    m = m / palm  # measured in palm sizes
    a, b = m[0, 5] - m[0, 0], m[0, 17] - m[0, 0]
    if a[0] * b[1] - a[1] * b[0] < 0:  # a left hand: mirror it so it looks like a right hand
        m[..., 0] *= -1
    return m.reshape(FRAMES, 63).astype(np.float32)

X = np.stack([move_features(move) for _, move in practice])
y = np.array([LABELS.index(label) for label, _ in practice])
print("X:", X.shape, "  y:", y.shape)

## 7. Make extra practice copies

About 90 moves is a small pile. So we make **10 slightly changed copies** of each one: tilted a little (up to 15°), a little bigger or smaller, faster at the start or the end, and a tiny bit shaky.

A J is still a J when it's tilted or quicker, so the copies teach the model what matters (the shape of the path) and what doesn't. This is called **data augmentation**.

In [ ]:
rng = np.random.default_rng(42)

def augment(features):
    """A slightly changed copy of one move: tilted, resized, sped up or slowed down, a bit shaky."""
    m = features.reshape(FRAMES, 21, 3).copy()
    angle = np.radians(rng.uniform(-15, 15))  # tilt by up to 15 degrees
    c, s = np.cos(angle), np.sin(angle)
    x, y_ = m[..., 0].copy(), m[..., 1].copy()
    m[..., 0], m[..., 1] = c * x - s * y_, s * x + c * y_
    m *= rng.uniform(0.85, 1.15)  # up to 15% bigger or smaller
    t = np.linspace(0, 1, FRAMES) ** rng.uniform(0.75, 1.33) * (FRAMES - 1)  # faster at the start or the end
    before = np.floor(t).astype(int)
    after = np.minimum(before + 1, FRAMES - 1)
    mix = (t - before)[:, None, None]
    m = m[before] * (1 - mix) + m[after] * mix
    m += rng.normal(0, 0.02, m.shape)  # a tiny bit of shake
    return m.reshape(FRAMES, 63).astype(np.float32)

X_train = np.concatenate([X] + [np.stack([augment(f) for f in X]) for _ in range(10)])
y_train = np.concatenate([y] * 11)
print("Training examples:", len(X_train), "(your", len(X), "moves + 10 changed copies of each)")

## 8. Build the model

The still-letter model looked at one set of numbers. This one looks at **24 frames in a row**, so it uses **Conv1D** layers.

Think of a Conv1D layer as a little window, 5 frames wide, that **slides along the move** looking for short patterns like "going straight down" or "curving left". Each of its 32 **filters** learns its own pattern.

| Layer | What it does |
|---|---|
| `Conv1D(32, 5)` | 32 pattern-finders, each looking at 5 frames at a time |
| `MaxPooling1D(2)` | keeps the stronger of every 2 frames, so the next layer sees a longer stretch (24 → 12 frames) |
| `Conv1D(32, 3)` | finds patterns of patterns: "down, *then* curve" |
| `GlobalAveragePooling1D()` | averages over time: how much of each pattern was in this move? |
| `Dropout(0.3)` | switches off 30% of the answers while training, so the model can't lean on just one |
| `Dense(3, softmax)` | 3 scores (J, Z, other) that add up to 1 |

In [ ]:
def make_model(batch_size=None):
    return tf.keras.Sequential([
        tf.keras.Input(shape=(FRAMES, 63), batch_size=batch_size),
        tf.keras.layers.Conv1D(32, 5, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling1D(2),
        tf.keras.layers.Conv1D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(len(LABELS), activation="softmax"),
    ])

tf.keras.utils.set_random_seed(42)  # same starting dials every run, so you get the same result
model = make_model()
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 9. Train

Same as before: 30 **epochs** (30 times through all the examples), 32 examples per **batch**.

In [ ]:
history = model.fit(X_train, y_train, epochs=30, batch_size=32, verbose=2)

## 10. The exam

Now the real test: your exam recording, which the model has **never seen**. The table shows what each move really was (rows) and what the model said (columns). Everything on the diagonal is right.

In [ ]:
exam = [(label, move) for label, move in moves_in(EXAM_FILE) if real_letter(label, move)]
X_exam = np.stack([move_features(move) for _, move in exam])
y_exam = np.array([LABELS.index(label) for label, _ in exam])

scores = model.predict(X_exam, verbose=0)
guess = scores.argmax(axis=1)
print(f"Exam score: {(guess == y_exam).mean():.0%} of {len(exam)} moves\n")
print(pd.crosstab(pd.Series([LABELS[i] for i in y_exam], name="really"),
                  pd.Series([LABELS[i] for i in guess], name="model said")))

## 11. How sure does it need to be?

The website only types a J or Z when the model is **at least 80% sure**. Otherwise it does nothing, which is better than typing a wrong letter.

In [ ]:
SURE = 0.8
sure = scores.max(axis=1) >= SURE
letter = guess != LABELS.index("other")
print(f"The model was at least {SURE:.0%} sure on {sure.mean():.0%} of the exam moves,")
print(f"and right on {(guess[sure] == y_exam[sure]).mean():.0%} of those.")
print("Times the website would have typed a wrong J or Z:", int((sure & letter & (guess != y_exam)).sum()))
print("Real J's and Z's it would have typed:", int((sure & letter & (guess == y_exam)).sum()),
      "of", int((y_exam != LABELS.index("other")).sum()))

## 12. Save the model for the website

Same trick as the still letters: the website needs a model that takes **exactly one move at a time** (shape `[1, 24, 63]`). The last line checks that the saved file gives the same answers as the model here.

In [ ]:
one_at_a_time = make_model(batch_size=1)  # the website needs a fixed shape: exactly 1 move
one_at_a_time.set_weights(model.get_weights())
converter = tf.lite.TFLiteConverter.from_keras_model(one_at_a_time)
with open("moves.tflite", "wb") as f:
    f.write(converter.convert())
with open("moves.json", "w") as f:
    json.dump(LABELS, f)

# Check: the saved file gives the same answers as the model here.
interpreter = tf.lite.Interpreter(model_path="moves.tflite")
interpreter.allocate_tensors()
inp, out = interpreter.get_input_details()[0], interpreter.get_output_details()[0]
same = 0
for features, expected in zip(X_exam, scores):
    interpreter.set_tensor(inp["index"], features[None])
    interpreter.invoke()
    same += np.allclose(interpreter.get_tensor(out["index"])[0], expected, atol=1e-4)
print("Input shape:", inp["shape"], " Saved model agrees on", same, "of", len(X_exam), "exam moves")

## 13. Download

This downloads `moves.tflite` (the model) and `moves.json` (the 3 labels). They go in your project at `public/models/moves/`.

In [ ]:
from google.colab import files
files.download("moves.tflite")
files.download("moves.json")